# Spatially tiled SpatialData in Celldega

Four Landscapes, two datasets x two storage layouts:

|  | DegaFiles (control) | SpatialData profile |
|---|---|---|
| **Pancreas** (8.1M transcripts, 377 genes) | 1 | 2 |
| **Prime skin** (74M transcripts, 5006 genes) | 3 | 4 |

The DegaFiles views are the controls. Reading across a row isolates the storage
layout; reading down a column isolates dataset scale. If a DegaFiles view is broken
then the reader changes regressed something, and that is the first thing to fix.

**Setup**

```bash
cd /Users/feni/Documents/spatial_tiles_SpatialData
source integration/.venv/bin/activate
jupyter lab integration/test_spatialdata_landscape.ipynb
```

Kernel: **spatial-tiles (integration)**. The HTTP server is started by the
next cell using celldega's own `get_local_server`. After any change under `celldega/js/`,
re-run `npm run build` in `celldega/` and restart the kernel.

In [ ]:
# Force the *local* widget bundle, with hot reload.
#
# celldega picks its front-end ESM at import time: a clean X.Y.Z install loads the
# published bundle from jsDelivr, so local js/ edits are invisible however often you
# rebuild or restart -- and nothing says so. ANYWIDGET_HMR selects the local bundle
# *and* turns on anywidget's hot reload, so `npm run build` refreshes the widget
# without restarting the kernel. Both must be set BEFORE celldega is imported.
#
# The spatial-tiles kernelspec also sets these, so this cell is belt and braces.
import os

os.environ["ANYWIDGET_HMR"] = "1"
os.environ["CELLDEGA_LOCAL_ESM"] = "1"

# celldega.pre imports pyvips, and this machine's homebrew libheif is linked against
# a libx265 that is not installed, so libvips writes a ~40 KB warning straight to
# fd 2, which Python-level redirection cannot intercept. Harmless -- nothing here
# touches HEIF. Permanent fix: `brew reinstall libheif`.
import contextlib
import sys


@contextlib.contextmanager
def _quiet_stderr():
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        sys.stderr.flush()
        os.dup2(devnull, 2)
        yield
    finally:
        sys.stderr.flush()
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


with _quiet_stderr():
    import celldega
    from celldega.viz import Landscape, get_local_server
    from celldega.viz.widget import _WIDGET_ESM

esm = str(_WIDGET_ESM)
local = esm.endswith(".js")
print(f"celldega {celldega.__version__}")
print("widget ESM:", ("LOCAL  " + esm) if local else ("CDN -- local js/ edits will NOT apply:\n  " + esm))
assert local, "set ANYWIDGET_HMR=1 before importing celldega, then restart the kernel"


In [ ]:
import json
import os
import time
from pathlib import Path
from urllib.request import Request, urlopen

WORKSPACE = Path.cwd().parent if Path.cwd().name == "integration" else Path.cwd()
DATA = WORKSPACE / "data"

# Celldega's own server, which serves the process CWD. It now honours byte ranges;
# before that fix SimpleHTTPRequestHandler ignored Range and returned whole files,
# which silently turned every row-group read into a full-file download.
os.chdir(DATA)

PORT = get_local_server()
time.sleep(0.5)
print(f"serving {DATA} at http://localhost:{PORT}")

PROFILE = "visualization/grid_files_v1"
URLS = {
    "pancreas_dega": f"http://localhost:{PORT}/pancreas_degafiles",
    "pancreas_sdata": f"http://localhost:{PORT}/pancreas_full.zarr/{PROFILE}",
    "skin_dega": f"http://localhost:{PORT}/skin_degafiles",
    "skin_sdata": f"http://localhost:{PORT}/skin_full.zarr/{PROFILE}",
}

for name, url in URLS.items():
    local = DATA / url.split(f"{PORT}/", 1)[1].rstrip("/")
    print(f"  {name:<16} {'ready' if (local / 'landscape_parameters.json').exists() else 'MISSING (still building?)'}")


In [ ]:
# What each dataset declares. The SpatialData rows should show a 'columns' projection
# and the display_* column names; the DegaFiles rows should show neither.
print(f"{'dataset':<16}{'tiles':>14}{'row groups':>12}{'genes':>8}  {'position col':<14}{'projection'}")
manifests = {}
for name, url in URLS.items():
    try:
        m = json.loads(urlopen(url + "/landscape_parameters.json", timeout=5).read())
    except Exception as exc:
        print(f"  {name:<16} unavailable: {type(exc).__name__}")
        continue
    manifests[name] = m
    g, trx = m["tile_grid"], m["row_group_files"]["transcripts"]
    cbg = m["row_group_files"].get("cbg", {})
    print(f"  {name:<16}{g['num_tiles_x']:>6} x{g['num_tiles_y']:>4}{trx['total_row_groups']:>12,}"
          f"{cbg.get('num_genes', len(cbg.get('gene_to_row_group', {}))):>8}  "
          f"{trx.get('position_column', 'geometry'):<14}{trx.get('columns', 'all columns')}")

In [ ]:
# The transport guarantees parquet-wasm depends on: 206 responses and a suffix range
# (how a reader locates the parquet footer before reading any row group).
chunk = URLS["pancreas_sdata"] + "/../../points/transcripts/points.parquet/chunk_00.parquet"
for label, header in [("prefix range", "bytes=0-7"), ("footer range", "bytes=-8")]:
    r = urlopen(Request(chunk, headers={"Range": header}))
    body = r.read()
    print(f"{label:14} {r.status} {r.headers.get('Content-Range')}  ({len(body)} bytes)")

assert r.status == 206, (
    "server ignored Range: row-group reads would download whole files. "
    "Check that celldega's CORSHTTPRequestHandler implements do_GET."
)


---
## 1. Pancreas — DegaFiles (control)

In [ ]:
Landscape(base_url=URLS["pancreas_dega"], technology="Xenium", ini_zoom=-6.5)

## 2. Pancreas — SpatialData profile

`base_url` points at the profile directory *inside* the zarr store. The manifest's
paths are relative (`../../points/transcripts/points.parquet`), so the reader resolves
back into the store with no knowledge of the zarr layout.

**Expect:** cell polygons at overview, DAPI underneath, transcripts on zoom-in,
cells recolouring when a gene is selected.

**Known gaps, not bugs:** hover labels come from `cell_code` (an integer index into the
table) and the profile does not yet populate `cats.nameMapping_inv`; there is also no
clustering or cell metadata, so category colouring will be empty.

In [ ]:
URLS["pancreas_sdata"]

In [ ]:
Landscape(base_url=URLS["pancreas_sdata"], technology="Xenium", ini_zoom=-6.5)

---
## 3. Prime skin — DegaFiles (control)

74M transcripts and 5006 genes: 9.2x the pancreas.

In [ ]:
Landscape(base_url=URLS["skin_dega"], technology="Xenium", ini_zoom=-6.5)

## 4. Prime skin — SpatialData profile

The scale case. Watch for whether panning stays responsive: `RowGroupTileReader`
caches only 4 reads and keys on the whole visible tile set, so every pan is currently
a full cache miss. That is a known limitation rather than a profile problem.

In [ ]:
Landscape(base_url=URLS["skin_sdata"], technology="Xenium", ini_zoom=-6.5)

---
## 5. The stores are still ordinary SpatialData stores

The point of tiling in place: nothing above needed a special reader, and nothing below
is disturbed by it.

In [ ]:
import spatialdata

for store in ("pancreas_full.zarr", "skin_full.zarr"):
    path = DATA / store
    if not path.exists():
        print(f"{store}: not built yet")
        continue
    sdata = spatialdata.read_zarr(path)
    pts = sdata.points["transcripts"]
    canonical = {"x", "y", "z", "feature_name", "cell_id", "transcript_id"} <= set(pts.columns)
    render = {"display_xy", "feature_code"} <= set(pts.columns)
    print(f"{store}: {len(pts):,} points, {len(sdata.shapes['cell_boundaries']):,} cells, "
          f"table {sdata.tables['table'].shape}")
    print(f"    canonical columns intact: {canonical} | render columns present: {render}")

In [ ]:
# The row grouping also makes spatial subsetting cheap from Python. This is the reason
# the tiling lives in the canonical file rather than in a viewer-only sidecar.
import pyarrow.parquet as pq

from spatialdata_io.experimental.regular_grid import RegularGrid

for store, key in (("pancreas_full.zarr", "pancreas_sdata"), ("skin_full.zarr", "skin_sdata")):
    if key not in manifests:
        continue
    m = manifests[key]
    trx = m["row_group_files"]["transcripts"]
    grid = RegularGrid.from_manifest_dict(m["tile_grid"])
    pdir = DATA / store / "points" / "transcripts" / "points.parquet"

    cx, cy = grid.num_tiles_x // 2, grid.num_tiles_y // 2
    tiles = [int(grid.tile_id(tx, ty)) for tx in range(cx, cx + 5) for ty in range(cy, cy + 4)]
    by_file = {}
    for t in tiles:
        fi, lo = grid.chunk_location(t, trx["max_row_groups_per_file"])
        by_file.setdefault(fi, []).append(lo)

    t0 = time.perf_counter()
    n = 0
    for fi, locals_ in by_file.items():
        n += pq.ParquetFile(pdir / trx["files"][fi]).read_row_groups(
            sorted(locals_), columns=["display_xy", "feature_code"]
        ).num_rows
    print(f"{store:<22}{len(tiles)} tiles -> {n:>9,} transcripts in "
          f"{(time.perf_counter() - t0) * 1000:6.1f} ms")

In [ ]:
test_url     = 'https://huggingface.co/datasets/cornhundred/celldega_Xenium_human_Pancreas_FFPE_v2/resolve/main/Xenium_V1_human_Pancreas_FFPE_outs_test'

In [ ]:
# sdata_hf_url = 'https://huggingface.co/datasets/cornhundred/SpatialData_with_spatial_tiles/resolve/main/pancreas_full.zarr/visualization/grid_files_v1'
sdata_hf_url = 'https://huggingface.co/datasets/cornhundred/SpatialData_with_spatial_tiles/resolve/main/skin_full.zarr/visualization/grid_files_v1'

In [ ]:
Landscape(base_url=test_url, technology="Xenium")

In [ ]:
Landscape(base_url=sdata_hf_url, technology="Xenium")

In [ ]:
# sd_tiles_gh = 'https://github.com/cornhundred/SpatialData_with_spatial_tiles/tree/main/skin_full/visualization/grid_files_v1'
# sd_tiles_gh = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_human_Pancreas_FFPE/main/Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp'
sd_tiles_gh = 'https://raw.githubusercontent.com/cornhundred/SpatialData_with_spatial_tiles/main/skin_full/visualization/grid_files_v1'

In [ ]:
Landscape(base_url=sd_tiles_gh, technology="Xenium")